In [5]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
#%matplotlib notebook
#%autosave 0
    
import logging
logging.basicConfig(format='%(asctime)s %(levelname)s:%(message)s', level=logging.DEBUG, datefmt='%I:%M:%S')

import os
import sys
import numpy as np
import pandas as pd
import sdata
import sdata.iolib
import sdata.iolib.hashtree
from sdata.iolib.hashtree import stable_cum_uuids_from_nested_list, stable_cum_uuid_from_nested_list

10:54:39 DEBUG:Loaded backend module://ipykernel.pylab.backend_inline version unknown.


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Autosave disabled


In [43]:
datatype = "mat1"
datatype_uuid = sdata.iolib.hashtree.generate_uuid_from_hash(datatype)
datatype_uuid

'83bb6f252fff55fdac7a4cf411145d94'

In [2]:
sdata.iolib.hashtree.NAMESPACE_SDATA

UUID('444bc445-2e5c-50c0-b758-d0195dad9867')

In [59]:
class ParameterSet:
    def __init__(self, name):
        self.name = name

In [3]:
nested_list = [[3.14159265, 42, "hello", 2.7182818], 
               [1, 1.2,"ssd"],
               ["a"], 
              ]
uid = stable_uuid_from_list([1,2,3])

In [4]:
uuids = stable_cum_uuids_from_nested_list(nested_list)
uuids

['f09e3370d8f953398366753ca7e16eb9',
 'b8f6eb73756c5d3d8f7456003bf0dc6d',
 'f6dd12b913135291b58bcd2f5f63f157']

In [7]:
uid = stable_cum_uuid_from_nested_list(nested_list)
uid

'f6dd12b913135291b58bcd2f5f63f157'

In [50]:
class TreeNode:
    def __init__(self, name, parent=None):
        self.name = name  # Der Name des Knotens
        self.children = {}  # Kinderknoten
        self.parent = parent  # Elternknoten

    def __lt__(self, other):
        """
        Ermöglicht die Sortierung von TreeNode-Objekten basierend auf ihrem Namen.
        """
        return self.name < other.name
    
    def get_child(self, path):
        """
        Navigiert den Pfad und gibt den Zielknoten zurück, falls vorhanden.
        :param path: Ein Pfad in der Form "child1/child2".
        :return: Der Zielknoten oder None, falls der Pfad nicht existiert.
        """
        path_list = path.strip("/").split("/")  # Pfad in eine Liste umwandeln
        current_node = self
        for part in path_list:
            if part in current_node.children:
                current_node = current_node.children[part]
            else:
                return None  # Pfad existiert nicht
        return current_node
        
    def add_path(self, path):
        path_list = path.strip("/").split("/")
        self.add_path_list(path_list)
            
    def add_path_list(self, path_list):
        if not path_list:
            return
        
        head, *tail = path_list
        if head not in self.children:
            self.children[head] = TreeNode(head, parent=self)
        self.children[head].add_path_list(tail)

    def repr(self, level=0):
        ret = "  " * level + repr(self.name) + "\n"
        for child in self.children.values():
            ret += child.__repr__(level + 1)
        return ret

    def __repr__(self):
        n_nodes = self.count_nodes()
        return f"({self.full_name()}|{n_nodes})"
    
    def __str__(self):
        n_nodes = self.count_nodes()
        return f"({self.full_name()}|{n_nodes})"
    
    def to_dict(self):
        """
        Konvertiert den Baum in ein verschachteltes Dictionary.
        """
        return {
            "name": self.name,
            "parent": self.parent,
            "children": {name: child.to_dict() for name, child in self.children.items()}
        }

    def to_json(self):
        """
        Exportiert den Baum in JSON-Format.
        """
        return json.dumps(self.to_dict(), indent=2)    
    
    def print_tree(self, padding="", is_last=True, is_root=True):
        """
        Druckt den Baum in einer visuellen Struktur mit Verzweigungen.
        :param padding: Aktuelle Einrückung.
        :param is_last: Ob der aktuelle Knoten das letzte Kind ist.
        :param is_root: Ob der aktuelle Knoten der Root-Knoten ist.
        """
        if is_root:
            print(self.name)  # Root-Knoten ohne Verzweigungssymbol
        else:
            branch = "└─" if is_last else "├─"
            print(padding + branch + self.name)
            padding += "   " if is_last else "│  "
        
        children = sorted(self.children.values())  # Alphabetische Sortierung
        for count, child in enumerate(children):
            child.print_tree(padding, count == (len(children) - 1), is_root=False)

    def full_name(self):
        """
        Gibt den vollständigen Pfad des Knotens zurück, von der Wurzel bis zu diesem Knoten.
        :return: Ein String, der den vollständigen Pfad darstellt.
        """
        if self.parent is None:
            return self.name  # Root-Knoten
        return f"{self.parent.full_name()}/{self.name}"
    
    def get_child_by_name(self, name):
        """
        Sucht im Baum nach einem Knoten mit dem angegebenen Namen.
        :param name: Der Name des gesuchten Knotens.
        :return: Der Knoten (TreeNode), wenn gefunden, sonst None.
        """
        if self.name == name:
            return self  # Aktueller Knoten passt
        
        for child in self.children.values():
            result = child.get_child_by_name(name)
            if result:
                return result  # Knoten gefunden
        
        return None  # Kein Knoten mit dem Namen gefunden

    def count_nodes(self):
        """
        Zählt die Anzahl aller Knoten im gesamten Teilbaum.
        :return: Die Anzahl aller Knoten (einschließlich des aktuellen Knotens).
        """
        count = 1  # Zähle den aktuellen Knoten
        for child in self.children.values():
            count += child.count_nodes()  # Rekursiv alle Kinder zählen
        return count
                    
def build_tree(hash_lists):
    root = TreeNode("root")
    for hash_list in hash_lists:
        root.add_path_list(hash_list)
    return root

tree = TreeNode("root")
tree.add_path("/tmp/a1/b2")
tree.add_path("/tmp/a3")
tree.add_path("/tmp/a2/b1/")
tree.add_path("/tmp/a2/b4/")
tree.to_dict()
print(tree)
tree.print_tree()


(root|8)
root
└─tmp
   ├─a1
   │  └─b2
   ├─a2
   │  ├─b1
   │  └─b4
   └─a3


In [55]:
tree = TreeNode(datatype_uuid)
#rootpath = ""
#paths = [rootpath] + uuids
#print(paths)
#path = os.path.join(*paths)
print(uuids)
tree.add_path_list(uuids)
tree.print_tree()


['f09e3370d8f953398366753ca7e16eb9', 'b8f6eb73756c5d3d8f7456003bf0dc6d', 'f6dd12b913135291b58bcd2f5f63f157']
83bb6f252fff55fdac7a4cf411145d94
└─f09e3370d8f953398366753ca7e16eb9
   └─b8f6eb73756c5d3d8f7456003bf0dc6d
      └─f6dd12b913135291b58bcd2f5f63f157


In [56]:
a2 = tree.get_child_by_name("b8f6eb73756c5d3d8f7456003bf0dc6d")
print(a2)
a2.print_tree()
a2.name

(83bb6f252fff55fdac7a4cf411145d94/f09e3370d8f953398366753ca7e16eb9/b8f6eb73756c5d3d8f7456003bf0dc6d|2)
b8f6eb73756c5d3d8f7456003bf0dc6d
└─f6dd12b913135291b58bcd2f5f63f157


'b8f6eb73756c5d3d8f7456003bf0dc6d'

In [57]:
a2.parent

(83bb6f252fff55fdac7a4cf411145d94/f09e3370d8f953398366753ca7e16eb9|3)

In [58]:
a2.to_dict()

{'name': 'b8f6eb73756c5d3d8f7456003bf0dc6d',
 'parent': (83bb6f252fff55fdac7a4cf411145d94/f09e3370d8f953398366753ca7e16eb9|3),
 'children': {'f6dd12b913135291b58bcd2f5f63f157': {'name': 'f6dd12b913135291b58bcd2f5f63f157',
   'parent': (83bb6f252fff55fdac7a4cf411145d94/f09e3370d8f953398366753ca7e16eb9/b8f6eb73756c5d3d8f7456003bf0dc6d|2),
   'children': {}}}}

In [54]:
b1 = tree.get_child("/tmp/a2/b1/")
b1.full_name()

AttributeError: 'NoneType' object has no attribute 'full_name'

In [132]:
b1.print_tree()

b1


In [129]:
print(b1)

root/tmp/a2/b1


In [126]:
tree.get_subtree_by_name("a1")

'a1'
  'b2'

In [110]:
tree = TreeNode("/")
tree.add_path("/tmp/a2/b4/")
tree

'/'
  'tmp'
    'a2'
      'b4'

In [111]:
tree = TreeNode("/")
tree.add_path_list([uid])
tree.add_path_list(uuids)


In [140]:
tree = TreeNode("root")
nested_list = [[3.14159265, 42, "hello", 2.7182818], 
               [1, 1.2,"ssd"],
               ["a"], 
              ]
nested_list2 = [[3.14159265, 42, "hello", 2.7182818], 
               [1, 2.2,"ssd"],
               ["a"],]

uuids = stable_uuids_from_nested_list(nested_list)
uuids2 = stable_uuids_from_nested_list(nested_list2)
tree.add_path_list(uuids)
tree.add_path_list(uuids2)
tree.print_tree()

root
└─f09e3370d8f953398366753ca7e16eb9
   ├─1514d36656d75f638a994b7fe87e2ad0
   │  └─68bb365dbf5c556faf848a6c8c72b205
   └─b0465a6b35ce5fd28338e04233c422d7
      └─68bb365dbf5c556faf848a6c8c72b205


In [5]:
def create_directories(node, base_path):
    current_path = os.path.join(base_path, node.name)
    if node.name != "root":  # Root nicht erstellen
        os.makedirs(current_path, exist_ok=True)
    for child in node.children.values():
        create_directories(child, current_path)

In [7]:

# Beispielnutzung
data = [3.14159265, 42, "hello", 2.7182818]
hashstr = stable_hash_from_list(data)
print(f"Stabiler Hash-Wert: {hashstr}")
stable_uuid_from_list(data)

Stabiler Hash-Wert: f54068f9b31cb94d56b454d36c1458877e5e45589684038b5c60097ad45b9052


UUID('f09e3370-d8f9-5339-8366-753ca7e16eb9')

In [143]:
# Beispiel-Hashwertlisten
hash_lists = [
    ["hash1", "hash2", "hash3"],
    ["hash1", "hash4"],
    ["hash5", "hash6"],
    ["hash1", "hash2", "hash7"]
]

# Baum erstellen
tree = build_tree(hash_lists)
print(tree)

root


In [142]:
base_directory = "/tmp/tree_structure"
create_directories(tree, base_directory)

NameError: name 'create_directories' is not defined

In [13]:
ls /tmp/tree_structure

root/


In [14]:
uuid.uuid4("ddf")

TypeError: uuid4() takes 0 positional arguments but 1 was given